# 1. Data Preparation
This [dataset](https://www.kaggle.com/datasets/rabieelkharoua/air-quality-and-health-impact-dataset) contains comprehensive information on the air quality and its impact on public health for 5,811 records. It includes variables such as air quality index (AQI), concentrations of various pollutants, weather conditions, and health impact metrics. The target variable is the health impact class, which categorizes the health impact based on the air quality and other related factors.

This dataset offers a comprehensive view of the relationship between air quality and public health, making it ideal for research, predictive modeling, and statistical analysis.

In [1]:
include("utils.jl")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.10/Project.toml`
  No Changes to `~/.julia/environments/v1.10/Manifest.toml`
┌ Info: Running `conda install -y -c anaconda conda` in root environment
└ @ Conda /home/pablo/.julia/packages/Conda/zReqD/src/Conda.jl:181


Channels:
 - anaconda
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/pablo/.julia/conda/3/x86_64

  added / updated specs:
    - conda


The following packages will be SUPERSEDED by a higher-priority channel:

  certifi            conda-forge/noarch::certifi-2024.8.30~ --> anaconda/linux-64::certifi-2024.8.30-py312h06a4308_0 



Preparing transaction: done
Verifying transaction: done
Executing transaction: done


┌ Info: Running `conda install -y -c conda-forge 'libstdcxx-ng>=3.4,<13.0'` in root environment
└ @ Conda /home/pablo/.julia/packages/Conda/zReqD/src/Conda.jl:181


Channels:
 - conda-forge
 - bioconda
 - defaults
 - anaconda
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/pablo/.julia/conda/3/x86_64

  added / updated specs:
    - libstdcxx-ng[version='>=3.4,<13.0']


The following packages will be SUPERSEDED by a higher-priority channel:

  certifi            anaconda/linux-64::certifi-2024.8.30-~ --> conda-forge/noarch::certifi-2024.8.30-pyhd8ed1ab_0 



Preparing transaction: done
Verifying transaction: done
Executing transaction: done


print_confusion_matrix (generic function with 2 methods)

In [2]:
using CSV, DataFrames

# Load the dataset from the 'dataset' folder
data = CSV.read("datasets/air_quality_health_impact_data.csv", DataFrame)

# Check the dataset
describe(data)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,RecordID,2906.0,1,2906.0,5811,0,Int64
2,AQI,248.438,0.00581738,249.128,499.859,0,Float64
3,PM10,148.655,0.0158481,147.635,299.902,0,Float64
4,PM2_5,100.224,0.0315489,100.506,199.985,0,Float64
5,NO2,102.293,0.00962478,102.988,199.98,0,Float64
6,SO2,49.4568,0.0110232,49.5302,99.9696,0,Float64
7,O3,149.312,0.001661,149.56,299.937,0,Float64
8,Temperature,14.9755,-9.991,14.9424,39.9634,0,Float64
9,Humidity,54.7769,10.0015,54.5439,99.9975,0,Float64


In [3]:
input_data = Matrix(data[!, 1:13]);
output_data = Int.(data[!, 15]);

@assert input_data isa Matrix
@assert output_data isa Vector{Int64}

To split the test from train we use the holdOut function with a fraction of 0.2. This means that 20% of the train data will be used for testing and the remaining 80% will be used for training.

In [4]:
# Split in train and test
(tr_idx, test_idx) = holdOut(size(input_data, 1), 0.2)

train_input = input_data[tr_idx,:]
train_output = output_data[tr_idx]
test_input = input_data[test_idx,:]
test_output = output_data[test_idx]

train_output = collect(train_output)
test_output = collect(test_output)

println("Train Input Size: ", size(train_input))
println("Train Output Size: ", size(train_output), " Categories:", sort(unique(train_output)))
println("Test Input Size: ", size(test_input))
println("Test Output Size: ", size(test_output), " Categories:", sort(unique(test_output)))


Train Input Size: (4649, 13)
Train Output Size: (4649,) Categories:[0, 1, 2, 3, 4]
Test Input Size: (1162, 13)
Test Output Size: (1162,) Categories:[0, 1, 2, 3, 4]


# 2. Definition of Models and Hyperparameters

In [5]:
## These are the modelsHyperParameters to be used in the models training

modelsHyperParameters = [
    # ANN configurations
    Dict("estimator" => :ANN, "topology" => (64,), "maxEpochs" => 200, "learningRate" => 0.01),
    Dict("estimator" => :ANN, "topology" => (128,), "maxEpochs" => 150, "learningRate" => 0.005),
    Dict("estimator" => :ANN, "topology" => (64, 32), "maxEpochs" => 100, "learningRate" => 0.01),
    Dict("estimator" => :ANN, "topology" => (128, 64), "maxEpochs" => 200, "learningRate" => 0.001),
    Dict("estimator" => :ANN, "topology" => (256,), "maxEpochs" => 300, "learningRate" => 0.0005),
    Dict("estimator" => :ANN, "topology" => (128, 64, 32), "maxEpochs" => 200, "learningRate" => 0.01),
    Dict("estimator" => :ANN, "topology" => (64, 64), "maxEpochs" => 250, "learningRate" => 0.005),
    Dict("estimator" => :ANN, "topology" => (256, 128), "maxEpochs" => 200, "learningRate" => 0.001),

    # SVM configurations
    Dict("estimator" => :SVM, "kernel" => "rbf", "C" => 1.0, "degree"=>2),
    Dict("estimator" => :SVM, "kernel" => "linear", "C" => 1.0, "degree"=>4),
    Dict("estimator" => :SVM, "kernel" => "poly", "C"=> 0.01 , "degree" => 2),
    Dict("estimator" => :SVM, "kernel" => "sigmoid", "C" => 1.0, "degree" => 3),
    Dict("estimator" => :SVM, "kernel" => "rbf", "C" => 0.01, "degree" => 3),
    Dict("estimator" => :SVM, "kernel" => "poly", "C"=> 5, "degree" => 3),
    Dict("estimator" => :SVM, "kernel" => "linear", "C"=> 0.1, "degree" => 3),
    Dict("estimator" => :SVM, "kernel" => "sigmoid", "C" => 10.0, "degree" => 5),

    # Decision Tree configurations
    Dict("estimator" => :DecisionTree, "max_depth" => 3, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 5, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 7, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 10, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 15, "random_state" => 42),
    Dict("estimator" => :DecisionTree, "max_depth" => 20, "random_state" => 42),

    # kNN configurations
    Dict("estimator" => :KNN, "k" => 3),
    Dict("estimator" => :KNN, "k" => 5),
    Dict("estimator" => :KNN, "k" => 7),
    Dict("estimator" => :KNN, "k" => 9),
    Dict("estimator" => :KNN, "k" => 11),
    Dict("estimator" => :KNN, "k" => 15)
]

# println(modelsHyperParameters)

28-element Vector{Dict{String, Any}}:
 Dict("maxEpochs" => 200, "learningRate" => 0.01, "estimator" => :ANN, "topology" => (64,))
 Dict("maxEpochs" => 150, "learningRate" => 0.005, "estimator" => :ANN, "topology" => (128,))
 Dict("maxEpochs" => 100, "learningRate" => 0.01, "estimator" => :ANN, "topology" => (64, 32))
 Dict("maxEpochs" => 200, "learningRate" => 0.001, "estimator" => :ANN, "topology" => (128, 64))
 Dict("maxEpochs" => 300, "learningRate" => 0.0005, "estimator" => :ANN, "topology" => (256,))
 Dict("maxEpochs" => 200, "learningRate" => 0.01, "estimator" => :ANN, "topology" => (128, 64, 32))
 Dict("maxEpochs" => 250, "learningRate" => 0.005, "estimator" => :ANN, "topology" => (64, 64))
 Dict("maxEpochs" => 200, "learningRate" => 0.001, "estimator" => :ANN, "topology" => (256, 128))
 Dict("estimator" => :SVM, "C" => 1.0, "kernel" => "rbf", "degree" => 2)
 Dict("estimator" => :SVM, "C" => 1.0, "kernel" => "linear", "degree" => 4)
 ⋮
 Dict("estimator" => :DecisionTree, "max_de

----
----

# 3. First approach: Using Min-Max Normalization.

In this approach we will use the Min-Max Normalization technique to normalize the data. And then, realize all the models training to finally compare the results.

We start with the normalization of the data. We use a function that normalize the train_input and test_input with the normalization parameters calculated on train_input.

In [6]:
train_input_minmax, test_input_minmax = normalizeData(train_input, test_input, :MinMax)

([0.10139438801859184 0.15193572200499816 … 0.2857142857142857 0.16666666666666666; 0.4814942330865898 0.6225146251757172 … 0.2857142857142857 0.16666666666666666; … ; 0.341022551213634 0.8028676872052769 … 0.21428571428571427 0.16666666666666666; 0.8383542778447237 0.38222421889757163 … 0.42857142857142855 0.0], [0.8710621449474952 0.9331303597428052 … 0.21428571428571427 0.25; 0.15252194870029265 0.652926408302796 … 0.35714285714285715 0.16666666666666666; … ; 0.9411258392150111 0.8585636887668742 … 0.5 0.0; 0.028059907040798762 0.5359205893232234 … 0.5 0.25])

In this approach we wont use crossValidation, so we will train the models on the whole dataset and then evaluate them on the test set.

Pre-normalization data characteristics:

In [7]:
describe(DataFrame(train_input, names(data)[1:13]))

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Float64,Float64,Float64,Int64,DataType
1,RecordID,2918.61,1.0,2931.0,5810.0,0,Float64
2,AQI,249.665,0.00581738,250.638,499.859,0,Float64
3,PM10,149.05,0.0158481,148.551,299.801,0,Float64
4,PM2_5,100.253,0.0315489,100.38,199.985,0,Float64
5,NO2,102.013,0.101998,102.838,199.98,0,Float64
6,SO2,49.4736,0.0110232,49.7001,99.9696,0,Float64
7,O3,149.5,0.001661,150.129,299.92,0,Float64
8,Temperature,15.0248,-9.991,14.9923,39.9634,0,Float64
9,Humidity,54.8483,10.0015,54.6595,99.9975,0,Float64


Post-normalization data with min-max:

In [8]:
describe(DataFrame(train_input_minmax, names(data)[1:13]))

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Float64,Float64,Float64,Int64,DataType
1,RecordID,0.502257,0.0,0.50439,1.0,0,Float64
2,AQI,0.499466,0.0,0.501411,1.0,0,Float64
3,PM10,0.497136,0.0,0.495474,1.0,0,Float64
4,PM2_5,0.501226,0.0,0.501857,1.0,0,Float64
5,NO2,0.509866,0.0,0.513991,1.0,0,Float64
6,SO2,0.49483,0.0,0.497097,1.0,0,Float64
7,O3,0.498466,0.0,0.500561,1.0,0,Float64
8,Temperature,0.500773,0.0,0.500121,1.0,0,Float64
9,Humidity,0.49832,0.0,0.496222,1.0,0,Float64


In [9]:
accuracies = []
models = []
for (idx, modelHyperParameters) in enumerate(modelsHyperParameters)
    model = genModel(modelHyperParameters)
    fit!(model, train_input_minmax, train_output)
    acc = score(model, test_input_minmax, test_output)
    push!(accuracies, (idx, modelHyperParameters["estimator"], acc))
    push!(models, deepcopy(model))
end;

/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

Once we have the accuracies, we will take the best of each model to create the ensemble, in this case we will create a stack, and then train the stack.

In [10]:
println(accuracies)
best_models_position = best_model_positions(accuracies)

stacking_classifier = StackingClassifier(
	estimators = [("m$(idx)_" * string(modelsHyperParameters[idx]["estimator"]), models[idx]) for idx in best_models_position],
	final_estimator = SVC(probability = true), n_jobs = -1)

fit!(stacking_classifier, train_input_minmax, train_output)

Any[(1, :ANN, 0.9345955249569707), (2, :ANN, 0.9268502581755593), (3, :ANN, 0.9380378657487092), (4, :ANN, 0.9449225473321858), (5, :ANN, 0.9225473321858864), (6, :ANN, 0.9337349397590361), (7, :ANN, 0.9371772805507745), (8, :ANN, 0.9371772805507745), (9, :SVM, 0.9010327022375215), (10, :SVM, 0.8932874354561101), (11, :SVM, 0.8227194492254734), (12, :SVM, 0.8201376936316696), (13, :SVM, 0.8227194492254734), (14, :SVM, 0.882960413080895), (15, :SVM, 0.8227194492254734), (16, :SVM, 0.7891566265060241), (17, :DecisionTree, 0.8373493975903614), (18, :DecisionTree, 0.8631669535283993), (19, :DecisionTree, 0.8769363166953529), (20, :DecisionTree, 0.8752151462994836), (21, :DecisionTree, 0.882960413080895), (22, :DecisionTree, 0.8838209982788297), (23, :KNN, 0.8356282271944923), (24, :KNN, 0.8364888123924269), (25, :KNN, 0.842512908777969), (26, :KNN, 0.8364888123924269), (27, :KNN, 0.8373493975903614), (28, :KNN, 0.8364888123924269)]


/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

PyObject StackingClassifier(estimators=[('m25_KNN', KNeighborsClassifier(n_neighbors=7)),
                               ('m9_SVM', SVC(degree=2)),
                               ('m4_ANN',
                                MLPClassifier(hidden_layer_sizes=(128, 64))),
                               ('m22_DecisionTree',
                                DecisionTreeClassifier(max_depth=20,
                                                       random_state=42))],
                   final_estimator=SVC(probability=True), n_jobs=-1)

Now we get the metrics of the stack ensemble.
After the best model selection and the stacking process, in some cases, we miss one of the clases in the output, so we decided to add 1 element of each class both in the train and in the test outputs.

In [11]:
outputs = stacking_classifier.predict(test_input_minmax)
categories = sort(unique(output_data))

x = copy(outputs)
y = vec(copy(test_output))

# We need to ensure that the confusion matrix recives the same categories:
if size(unique(x),1) != size(unique(y),1)
    append!(x, categories)
    append!(y, categories)
end

accuracy, er, recall, specificity, ppv, npv, f_score, cm = confusionMatrix(x, y)


(0.9773778920308483, 0.02262210796915165, 0.9643605315851733, 0.03563946841482676, 0.9448792892538722, 0.9573158952117332, 0.9434447300771208, [948 14 … 5 7; 9 96 … 5 4; … ; 0 0 … 8 0; 0 0 … 0 1])

And print the metrics for this First Approach:

In [12]:
# Display results
println("METRICS 1st APPROACH (Min-Max Normalization):")
println("---------------------")
println("Accuracy: ", accuracy)
println("Error Rate: ", er)
println("Sensitivity (Recall): ", recall)
println("Specificity: ", specificity)
println("Precision: ", ppv)
println("Negative Predictive Value: ", npv)
println("F-Score: ", f_score)
println("----------------------")
println()

print_confusion_matrix(cm, ["Class 0", "Class 1", "Class 2", "Class 3", "Class 4"])

METRICS 1st APPROACH (Min-Max Normalization):
---------------------
Accuracy: 0.9773778920308483
Error Rate: 0.02262210796915165
Sensitivity (Recall): 0.9643605315851733
Specificity: 0.03563946841482676
Precision: 0.9448792892538722
Negative Predictive Value: 0.9573158952117332
F-Score: 0.9434447300771208
----------------------

CONFUSION MATRIX:
             Class 0   Class 1   Class 2   Class 3   Class 4
   Class 0       948        14         9         5         7
   Class 1         9        96         7         5         4
   Class 2         0         2        48         3         1
   Class 3         0         0         0         8         0
   Class 4         0         0         0         0         1


## In this point we should make the analisys of the results

----
----

# 4. Second Approach: Using Standard normalization

In this approach we will use the Zero-Mean Normalization technique to normalize the data. And then, train the models to finally compare the results.

We start with the normalization of the data. We use a function that normalize the train_input and test_input with the normalization parameters calculated on train_input.

In [13]:
train_input_zm, test_input_zm = normalizeData(train_input, test_input, :MinMax)

([0.10139438801859184 0.15193572200499816 … 0.2857142857142857 0.16666666666666666; 0.4814942330865898 0.6225146251757172 … 0.2857142857142857 0.16666666666666666; … ; 0.341022551213634 0.8028676872052769 … 0.21428571428571427 0.16666666666666666; 0.8383542778447237 0.38222421889757163 … 0.42857142857142855 0.0], [0.8710621449474952 0.9331303597428052 … 0.21428571428571427 0.25; 0.15252194870029265 0.652926408302796 … 0.35714285714285715 0.16666666666666666; … ; 0.9411258392150111 0.8585636887668742 … 0.5 0.0; 0.028059907040798762 0.5359205893232234 … 0.5 0.25])

Post-normalization data with zero-mean:

In [14]:
describe(DataFrame(train_input_zm, names(data)[1:13]))

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Float64,Float64,Float64,Int64,DataType
1,RecordID,0.502257,0.0,0.50439,1.0,0,Float64
2,AQI,0.499466,0.0,0.501411,1.0,0,Float64
3,PM10,0.497136,0.0,0.495474,1.0,0,Float64
4,PM2_5,0.501226,0.0,0.501857,1.0,0,Float64
5,NO2,0.509866,0.0,0.513991,1.0,0,Float64
6,SO2,0.49483,0.0,0.497097,1.0,0,Float64
7,O3,0.498466,0.0,0.500561,1.0,0,Float64
8,Temperature,0.500773,0.0,0.500121,1.0,0,Float64
9,Humidity,0.49832,0.0,0.496222,1.0,0,Float64


In [15]:
accuracies = []
models = []
for (idx, modelHyperParameters) in enumerate(modelsHyperParameters)
    model = genModel(modelHyperParameters)
    fit!(model, train_input_zm, train_output)
    acc = score(model, test_input_zm, test_output)
    push!(accuracies, (idx, modelHyperParameters["estimator"], acc))
    push!(models, deepcopy(model))
end;

/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (150) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/pablo/Dropbox/MIA/ML1/sk/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

In [16]:
println(accuracies)
best_models_position = best_model_positions(accuracies)

stacking_classifier = StackingClassifier(
	estimators = [("m$(idx)_" * string(modelsHyperParameters[idx]["estimator"]), models[idx]) for idx in best_models_position],
	final_estimator = SVC(probability = true), n_jobs = -1)

fit!(stacking_classifier, train_input_zm, train_output)

Any[(1, :ANN, 0.9406196213425129), (2, :ANN, 0.9259896729776248), (3, :ANN, 0.9380378657487092), (4, :ANN, 0.9380378657487092), (5, :ANN, 0.9268502581755593), (6, :ANN, 0.9423407917383821), (7, :ANN, 0.9345955249569707), (8, :ANN, 0.9414802065404475), (9, :SVM, 0.9010327022375215), (10, :SVM, 0.8932874354561101), (11, :SVM, 0.8227194492254734), (12, :SVM, 0.8201376936316696), (13, :SVM, 0.8227194492254734), (14, :SVM, 0.882960413080895), (15, :SVM, 0.8227194492254734), (16, :SVM, 0.7891566265060241), (17, :DecisionTree, 0.8373493975903614), (18, :DecisionTree, 0.8631669535283993), (19, :DecisionTree, 0.8769363166953529), (20, :DecisionTree, 0.8752151462994836), (21, :DecisionTree, 0.882960413080895), (22, :DecisionTree, 0.8838209982788297), (23, :KNN, 0.8356282271944923), (24, :KNN, 0.8364888123924269), (25, :KNN, 0.842512908777969), (26, :KNN, 0.8364888123924269), (27, :KNN, 0.8373493975903614), (28, :KNN, 0.8364888123924269)]


PyObject StackingClassifier(estimators=[('m25_KNN', KNeighborsClassifier(n_neighbors=7)),
                               ('m9_SVM', SVC(degree=2)),
                               ('m6_ANN',
                                MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                                              learning_rate_init=0.01)),
                               ('m22_DecisionTree',
                                DecisionTreeClassifier(max_depth=20,
                                                       random_state=42))],
                   final_estimator=SVC(probability=True), n_jobs=-1)

In [17]:
outputs = stacking_classifier.predict(test_input_zm)
categories = sort(unique(output_data))

x = copy(outputs)
y = vec(copy(test_output))

# We need to ensure that the confusion matrix recives the same categories:
if size(unique(x),1) != size(unique(y),1)
    append!(x, categories)
    append!(y, categories)
end

accuracy, er, recall, specificity, ppv, npv, f_score, cm = confusionMatrix(x, y)

(0.9307626392459297, 0.06923736075407028, 0.9265813880573233, 0.07341861194267668, 0.807022123958896, 0.9251746678569532, 0.8269065981148245, [945 32 … 9 10; 0 7 … 0 1; … ; 0 0 … 11 0; 0 0 … 0 1])

In [18]:
# Display results
println("METRICS 2nd APPROACH (Zero-Mean Normalization):")
println("---------------------")
println("Accuracy: ", accuracy)
println("Error Rate: ", er)
println("Sensitivity (Recall): ", recall)
println("Specificity: ", specificity)
println("Precision: ", ppv)
println("Negative Predictive Value: ", npv)
println("F-Score: ", f_score)
println("----------------------")
println()

print_confusion_matrix(cm, ["Class 0", "Class 1", "Class 2", "Class 3", "Class 4"])

METRICS 2nd APPROACH (Zero-Mean Normalization):
---------------------
Accuracy: 0.9307626392459297
Error Rate: 0.06923736075407028
Sensitivity (Recall): 0.9265813880573233
Specificity: 0.07341861194267668
Precision: 0.807022123958896
Negative Predictive Value: 0.9251746678569532
F-Score: 0.8269065981148245
----------------------

CONFUSION MATRIX:
             Class 0   Class 1   Class 2   Class 3   Class 4
   Class 0       945        32        13         9        10
   Class 1         0         7        48         0         1
   Class 2        12        73         1         1         1
   Class 3         0         0         2        11         0
   Class 4         0         0         0         0         1


----
----

# 5. Third Approach: Using Crossvalidation

In this approach we will use the Crossvalidation technique to to train the models, and using the normalization of the data that gave us the best results in the previous steps.